In [ ]:
import sys; sys.path.append('..'); sys.path.append('../../');  sys.path.append('../../gmsh')

In [ ]:
import inflation
import numpy as np

In [ ]:
def get_bsi_info_from_alpha_beta(alpha, beta):
    bsi_info = [np.array([alpha**2 + beta, alpha + beta**2, alpha * beta, alpha + beta, alpha + beta]),
            np.array([2 * alpha, 1, beta, 1, 1]),
            np.array([1, 2 * beta, alpha, 1, 1]),
            np.array([2, 0, 0, 0, 0]),
            np.array([0, 2, 0, 0, 0]),
            np.array([0, 0, 1, 0, 0])]
    return bsi_info

In [ ]:
# compute the integrand: sum coefficient[i] * term_i
# term_1 = k1 Cos[\[Theta]]^2 + k2 Sin[\[Theta]]^2)^2 * Cos[\[Delta] + \[Theta] ]^2 *Sin[\[Delta] + \[Theta] ]^2
# term_2 = (k1 Cos[\[Theta]]^2 + k2 Sin[\[Theta]]^2)^2 * Cos[\[Delta] + \[Theta]]^3 * Sin[\[Delta] + \[Theta]]
# term_3 = (k1 Cos[\[Theta]]^2 + k2 Sin[\[Theta]]^2)^2 * Cos[\[Delta] + \[Theta]] * Sin[\[Delta] + \[Theta]]^3
# term_4 = (k1 Cos[\[Theta]]^2 + k2 Sin[\[Theta]]^2)^2 * Cos[\[Delta] + \[Theta]]^4
# term_5 = (k1 Cos[\[Theta]]^2 + k2 Sin[\[Theta]]^2)^2 * Sin[\[Delta] + \[Theta]]^4
def integrateTheta(delta, k1, k2, coefficient, activation, res):
    theta = np.linspace(0, np.pi * 2, res)
    step = np.pi * 2 / (res - 1)
    term_1 = (k1 * np.cos(theta)**2 + k2 * np.sin(theta)**2)**2 * np.cos(delta + theta + np.pi / 2)**2 * np.sin(delta + theta + np.pi / 2)**2
    term_2 = (k1 * np.cos(theta)**2 + k2 * np.sin(theta)**2)**2 * np.cos(delta + theta + np.pi / 2)**3 * np.sin(delta + theta + np.pi / 2)
    term_3 = (k1 * np.cos(theta)**2 + k2 * np.sin(theta)**2)**2 * np.cos(delta + theta + np.pi / 2) * np.sin(delta + theta + np.pi / 2)**3
    term_4 = (k1 * np.cos(theta)**2 + k2 * np.sin(theta)**2)**2 * np.cos(delta + theta + np.pi / 2)**4
    term_5 = (k1 * np.cos(theta)**2 + k2 * np.sin(theta)**2)**2 * np.sin(delta + theta + np.pi / 2)**4
    integrand = activation[0] * coefficient[0] * term_1 + activation[1] * coefficient[1] * term_2 + activation[2] * coefficient[2] * term_3 + activation[3] * coefficient[3] * term_4 + activation[4] * coefficient[4] * term_5
    return np.sum(integrand) * step - (integrand[0] + integrand[1]) * step / 2.

### Validate objective

In [ ]:
alpha= 1.22262
beta= 1.09941
kappaAngle= -0.556292
psi= -0.0651634
k1= 3.23741
k2= 0.394415
# bending energy: 42.4718
# psi_a_b= -19.3392
#  21.5973
#  11.6349

In [ ]:
bsiInfo = get_bsi_info_from_alpha_beta(alpha, beta)

In [ ]:
coefficients = bsiInfo[0]

In [ ]:
activation = np.ones(5)
# activation[0] = 1

In [ ]:
delta = kappaAngle - psi

In [ ]:
integrateTheta(delta, k1, k2, coefficients, activation, 20000)

In [ ]:
bsis = inflation.BendingStiffnessIntegralSensitivity()

In [ ]:
bsis.update_delta_q(delta, k1, k2, coefficients, activation)

In [ ]:
bsis.objective

### Validate delta q

In [ ]:
bsis.update_delta_q(delta, k1, k2, coefficients, activation)

In [ ]:
bsis.objective

In [ ]:
bsis.delta_q_gradient

In [ ]:
bsis.delta_q_hessian

In [ ]:
class bending_stiffness_integral_class:
    def __init__(self, bsis):
        self.bsis = bsis
        self.vars = np.array([delta, *coefficients])

    def setVars(self, curr_vars):
        # self.bsis.update(curr_vars[0], 2, 1, np.ones(5), activation)
        self.bsis.update_delta_q(curr_vars[0], k1, k2, curr_vars[1:], activation)
        self.vars = curr_vars
        
    def numVars(self):
        return 6

    def getVars(self):
        return self.vars

    def energy(self): 
        return self.bsis.objective

    def gradient(self):
        return self.bsis.delta_q_gradient.copy()

    def hessian(self): 
        return self.bsis.delta_q_hessian.copy()
    
    def secondDerivative(self): 
        return self.bsis.delta_q_hessian[0][0]


In [ ]:
import fd_validation

In [ ]:
bsis_wrapper = bending_stiffness_integral_class(bsis)

In [ ]:
bsis_wrapper.energy()

In [ ]:
fd_validation.gradConvergencePlot(bsis_wrapper)

In [ ]:
import importlib
importlib.reload(fd_validation)

In [ ]:
fd_validation.hessConvergencePlot(bsis_wrapper, testHessVec=False)

In [ ]:
var_types = ['delta', 'coefficients']
var_indices = {'delta': [0],
               'coefficients': [1, 2, 3, 4, 5]}

In [ ]:
fd_validation.hessian_convergence_block_plot(bsis_wrapper, var_types, var_indices)

### Validate psi a b

In [ ]:
bsis.update(kappaAngle - psi, k1, k2, bsiInfo)

In [ ]:
bsis.objective

In [ ]:
bsis.psi_a_b_gradient

In [ ]:
bsis.psi_a_b_hessian

In [ ]:
class bending_stiffness_integral_class:
    def __init__(self, bsis):
        self.bsis = bsis
        self.vars = np.array([psi, alpha, beta])

    def setVars(self, curr_vars):
        curr_alpha = curr_vars[1]
        curr_beta = curr_vars[2]
        psi = curr_vars[0]
        bsi_info = get_bsi_info_from_alpha_beta(curr_alpha, curr_beta)
        bsis.update(kappaAngle - psi, k1, k2, bsi_info)        
        self.vars = curr_vars
        
    def numVars(self):
        return 3

    def getVars(self):
        return self.vars

    def energy(self): 
        return self.bsis.objective

    def gradient(self):
        return self.bsis.psi_a_b_gradient.copy()

    def hessian(self): 
        return self.bsis.psi_a_b_hessian.copy()
    
    def secondDerivative(self): 
        return self.bsis.psi_a_b_hessian[0][0]


In [ ]:
import fd_validation

In [ ]:
bsis_wrapper = bending_stiffness_integral_class(bsis)

In [ ]:
bsis_wrapper.energy()

In [ ]:
fd_validation.gradConvergencePlot(bsis_wrapper)

In [ ]:
import importlib
importlib.reload(fd_validation)

In [ ]:
fd_validation.hessConvergencePlot(bsis_wrapper, testHessVec=False)

In [ ]:
var_types = ['psi', 'alpha', 'beta']
var_indices = {'psi': [0],
               'alpha': [1],
                'beta': [2]}

In [ ]:
fd_validation.hessian_convergence_block_plot(bsis_wrapper, var_types, var_indices)